In [3]:
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql import SparkSession


In [1]:
import os
import sys
import warnings
import pyspark.pandas as ps

# Critical fix for virtualenv on Windows
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Increase timeouts significantly (helps a lot on Windows)
os.environ["SPARK_NETWORK_TIMEOUT"] = "1000s"
os.environ["SPARK_EXECUTOR_HEARTBEAT_INTERVAL"] = "120s"

# Optional: extra configs for stability
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.python.worker.reuse=true pyspark-shell"

# Tell PyArrow to ignore timezone metadata
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

# Suppress only this specific warning
warnings.filterwarnings("ignore", category=ps.utils.PandasAPIOnSparkAdviceWarning)

E:\project-workspace\quarto-book\data-analysis-model-book\.venv\Lib\site-packages\pyspark\pandas\__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC Taxi") \
    .config("spark.network.timeout", "1000s") \
    .config("spark.executor.heartbeatInterval", "120s") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

import pyspark.pandas as ps
#ps.set_option("compute.ops_on_diff_frames", True)

## Object Creation

In [9]:
s = ps.Series([1, 3, 5, np.nan, 6, 8])
s

0    1.0
1    3.0
2    5.0
3    NaN
4    6.0
5    8.0
dtype: float64

In [3]:
from pyspark.sql.functions import broadcast, col, avg, count, round

psdf = ps.read_parquet(
    "../data/taxi/yellow_tripdata_2023-01.parquet"
)

# Engineer features with Pandas syntax
psdf["fare_per_mile"] = (
    psdf["fare_amount"] / psdf["trip_distance"]
).round(2)

# Convert back to native Spark DataFrame for the broadcast join
sdf = psdf.to_spark()

# Use native Spark for the performance-critical join
zones_sdf = spark.createDataFrame(
    ps.read_csv(
        "../data/taxi/taxi_zone_lookup.csv"
    ).to_pandas()
)

result = (
    sdf.filter(col("trip_distance") > 0)
       .join(
           broadcast(zones_sdf.select(
               col("LocationID").alias("PULocationID"),
               col("Borough").alias("pickup_borough")
           )),
           on="PULocationID",
           how="left"
       )
       .groupBy("pickup_borough")
       .agg(
           count("*").alias("n_trips"),
           round(avg("fare_per_mile"), 2).alias("avg_fare_per_mile")
       )
       .orderBy("n_trips", ascending=False)
)

result.show()

+--------------+-------+-----------------+
|pickup_borough|n_trips|avg_fare_per_mile|
+--------------+-------+-----------------+
|     Manhattan|2686017|             9.09|
|        Queens| 276224|            10.61|
|       Unknown|  38568|            15.29|
|      Brooklyn|  16069|            14.83|
|         Bronx|   3149|             18.9|
|           N/A|    525|           334.95|
| Staten Island|    267|            10.01|
|           EWR|     85|          2125.22|
+--------------+-------+-----------------+

